In [9]:
import pandas as pd
import sqlite3
import warnings
warnings.filterwarnings('ignore')

## 1

In [10]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

pageviews=pd.read_sql_query("SELECT * FROM pageviews;", conn)
pageviews

,index,uid,datetime
0,0,admin_1,2020-04-17 12:01:08.463179
1,1,admin_1,2020-04-17 12:01:23.743946
2,2,admin_3,2020-04-17 12:17:39.287778
3,3,admin_3,2020-04-17 12:17:40.001768
4,4,admin_1,2020-04-17 12:27:30.646665
...,...,...,...
1075,1075,user_25,2020-05-21 23:23:49.995349
1076,1076,admin_1,2020-05-21 23:49:22.386789
1077,1077,admin_1,2020-05-22 10:36:14.662600
1078,1078,user_5,2020-05-22 11:30:18.368990


In [11]:

checker=pd.read_sql_query("SELECT * FROM checker;", conn)
checker

,index,status,success,timestamp,numTrials,labname,uid
0,0,checking,0,2020-04-16 21:12:50.740474,5,None,admin_1
1,1,ready,0,2020-04-16 21:12:54.708365,5,code_rvw,admin_1
2,2,checking,0,2020-04-16 21:46:47.769088,7,None,admin_1
3,3,ready,0,2020-04-16 21:46:48.121217,7,lab02,admin_1
4,4,checking,0,2020-04-16 21:53:01.862637,6,code_rvw,admin_1
...,...,...,...,...,...,...,...
3397,3397,ready,0,2020-05-21 20:19:06.872761,7,laba06s,user_1
3398,3398,checking,0,2020-05-21 20:22:41.785725,8,laba06s,user_1
3399,3399,ready,0,2020-05-21 20:22:41.877806,8,laba06s,user_1
3400,3400,checking,0,2020-05-21 20:37:00.129678,9,laba06s,user_1


## 2

In [12]:
del_query = "DROP TABLE IF EXISTS datamart;"
conn.execute(del_query)

query="""
CREATE TABLE datamart AS
SELECT pv.uid, ch.labname, ch.timestamp AS first_commit_ts, pv.datetime AS first_view_ts
FROM pageviews AS pv
FULL JOIN checker AS ch
ON pv.uid = ch.uid AND pv.uid IN (
    SELECT uid 
    FROM checker 
    WHERE status = 'ready' 
    AND numTrials=1 AND labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
)
GROUP BY ch.uid, ch.labname;
"""
del_query="""
DELETE datamart;
"""

conn.execute(query)
conn.commit()
datamart=pd.read_sql_query("SELECT * FROM datamart;", conn, parse_dates=["first_commit_ts", "first_view_ts"])
datamart

,uid,labname,first_commit_ts,first_view_ts
0,admin_1,None,NaT,2020-04-17 12:01:08.463179
1,None,laba04,2020-04-26 09:07:46.355781,NaT
2,None,laba04s,2020-04-26 09:08:56.887761,NaT
3,None,laba06,2020-05-19 11:15:09.236126,NaT
4,None,laba06s,2020-05-19 11:25:54.074898,NaT
...,...,...,...,...
201,None,laba04s,2020-04-19 10:22:35.761944,NaT
202,None,laba05,2020-05-02 13:28:07.705193,NaT
203,None,laba06,2020-05-16 17:56:15.755553,NaT
204,None,laba06s,2020-05-16 20:01:07.900727,NaT


## 3

In [13]:
test=datamart[datamart['first_view_ts'].notna()]
control=datamart[datamart['first_view_ts'].isna()]

avg=test['first_view_ts'].mean()

control['first_view_ts'].fillna(avg, inplace=True)

In [14]:
control

,uid,labname,first_commit_ts,first_view_ts
1,None,laba04,2020-04-26 09:07:46.355781,2020-04-26 17:24:49.174370048
2,None,laba04s,2020-04-26 09:08:56.887761,2020-04-26 17:24:49.174370048
3,None,laba06,2020-05-19 11:15:09.236126,2020-04-26 17:24:49.174370048
4,None,laba06s,2020-05-19 11:25:54.074898,2020-04-26 17:24:49.174370048
5,None,project1,2020-05-11 16:11:07.874291,2020-04-26 17:24:49.174370048
...,...,...,...,...
201,None,laba04s,2020-04-19 10:22:35.761944,2020-04-26 17:24:49.174370048
202,None,laba05,2020-05-02 13:28:07.705193,2020-04-26 17:24:49.174370048
203,None,laba06,2020-05-16 17:56:15.755553,2020-04-26 17:24:49.174370048
204,None,laba06s,2020-05-16 20:01:07.900727,2020-04-26 17:24:49.174370048


In [ ]:
test.to_sql('test', conn, if_exists='replace', index=False)
control.to_sql('control', conn, if_exists='replace', index=False)



118

In [16]:
conn.close()